# SAGE / DisjunctiveNet — Kaggle reproducibility runner

This notebook reproduces the current certified Julia baseline, controlled discontinuity-learning test, and multi-branch matched-budget gate. It deliberately makes **no real-dataset performance claim**.

Kaggle setup: turn **Internet on**. A GPU is not needed; the current solver path is CPU-bound. Each stage fails loudly if its acceptance token is absent.

In [ ]:
from pathlib import Path
import json, os, platform, re, shutil, subprocess, sys, tarfile, time, urllib.request

REPO_URL = 'https://github.com/ghugn/SAGE_ICML2027.git'
REPO_BRANCH = 'main'
JULIA_VERSION = '1.13.3'  # matches the committed Manifest.toml and local verification
RUN_FULL_TESTS = True

ON_KAGGLE = Path('/kaggle/working').exists()
WORK = Path('/kaggle/working') if ON_KAGGLE else Path.cwd()
DEPOT = WORK / 'julia_depot'
LOG_DIR = WORK / 'sage_reproduction_logs'
DEPOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
STAGE_RESULTS = {}
memory_kib = None
if Path('/proc/meminfo').exists():
    match = re.search(r'^MemTotal:\s+(\d+)', Path('/proc/meminfo').read_text(), re.MULTILINE)
    memory_kib = int(match.group(1)) if match else None
environment = {
    'work': str(WORK), 'on_kaggle': ON_KAGGLE, 'python': sys.version.split()[0],
    'platform': platform.platform(), 'machine': platform.machine(),
    'cpu_count': os.cpu_count(), 'memory_kib': memory_kib,
}
(LOG_DIR / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(environment)

## 1. Install or locate Julia
The archive is downloaded from Julia's official S3 distribution if the executable is not already available.

In [ ]:
def detected_julia_version(executable):
    try:
        return subprocess.check_output([str(executable), '--startup-file=no', '-e', 'print(VERSION)'], text=True).strip()
    except (OSError, subprocess.CalledProcessError):
        return None

def locate_or_install_julia(version=JULIA_VERSION):
    existing = shutil.which('julia')
    if existing and detected_julia_version(existing) == version:
        return Path(existing)
    if existing:
        print(f'Ignoring system Julia {detected_julia_version(existing)} at {existing}; reproducibility requires {version}')
    install_root = WORK / f'julia-{version}'
    executable = install_root / 'bin' / 'julia'
    if executable.exists() and detected_julia_version(executable) == version:
        return executable
    minor = '.'.join(version.split('.')[:2])
    archive = WORK / f'julia-{version}-linux-x86_64.tar.gz'
    url = f'https://julialang-s3.julialang.org/bin/linux/x64/{minor}/julia-{version}-linux-x86_64.tar.gz'
    print('Downloading', url)
    urllib.request.urlretrieve(url, archive)
    with tarfile.open(archive, 'r:gz') as bundle:
        bundle.extractall(WORK)
    assert executable.exists(), f'Julia executable not found at {executable}'
    return executable

JULIA = locate_or_install_julia()
print('Julia:', JULIA)
actual_julia_version = detected_julia_version(JULIA)
print('Julia version:', actual_julia_version)
assert actual_julia_version == JULIA_VERSION, (actual_julia_version, JULIA_VERSION)

## 2. Get the exact repository
When the notebook is run from a checked-out repository it uses that checkout. Otherwise it clones `main` into Kaggle's writable directory.

In [ ]:
if (Path.cwd() / 'Project.toml').exists():
    REPO = Path.cwd().resolve()
else:
    REPO = WORK / 'SAGE_ICML2027'
    if not (REPO / '.git').exists():
        subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
print('Repository:', REPO)
subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], check=True)

## 3. Instantiate the Julia environment
The first run can take several minutes because Julia compiles packages. Logs are retained in `/kaggle/working/sage_reproduction_logs`.

In [ ]:
BASE_ENV = os.environ.copy()
BASE_ENV['JULIA_DEPOT_PATH'] = str(DEPOT)
BASE_ENV['JULIA_NUM_THREADS'] = str(max(1, os.cpu_count() or 1))

def run_stage(name, args, required_token=None):
    command = [str(JULIA), '--project=.', *args]
    print('\n$', ' '.join(command), flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(command, cwd=REPO, env=BASE_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code = process.wait()
    elapsed = time.perf_counter() - started
    output = ''.join(lines)
    (LOG_DIR / f'{name}.log').write_text(output, encoding='utf-8')
    token_ok = required_token is None or required_token in output
    STAGE_RESULTS[name] = {'exit_code': code, 'seconds': round(elapsed, 3), 'required_token': required_token, 'token_ok': token_ok}
    (LOG_DIR / 'stage_results.json').write_text(json.dumps(STAGE_RESULTS, indent=2), encoding='utf-8')
    shutil.make_archive(str(WORK / 'sage_reproduction_bundle'), 'zip', root_dir=LOG_DIR)
    print(f'[{name}] exit={code} elapsed={elapsed:.3f}s token_ok={token_ok}', flush=True)
    if code != 0:
        raise RuntimeError(f'{name} failed with exit code {code}')
    if not token_ok:
        raise AssertionError(f'{name} did not emit {required_token!r}')
    return output

instantiate_log = run_stage('00_instantiate', ['-e', 'using Pkg; Pkg.instantiate(); Pkg.precompile()'])

## 4. Foundation and estimator verification

In [ ]:
foundation_log = run_stage('01_foundations', ['scripts/verify_foundations.jl'], 'FOUNDATION_VERIFICATION=PASS')
sage_log = run_stage('02_sage_gradient', ['scripts/verify_sage_gradient.jl'], 'SAGE_GRADIENT_VERIFICATION=PASS')

## 5. Controlled learning-through-discontinuity test
This is the current go/no-go gate before a larger Flux or dataset experiment. The detailed assumptions and limitations are in `docs/research_log/2026-11-13_phase5_discontinuity_learning.md`.

In [ ]:
learning_log = run_stage('03_discontinuity_learning', ['scripts/benchmark_discontinuity_learning.jl'], 'DISCONTINUITY_LEARNING_GO_NO_GO=PASS')

## 6. Multi-branch matched-budget benchmark
This proposal-aligned E1-E3 slice measures analytic/backend parity, gradient fidelity, certificate/pruning coverage, global solver calls, and complete backward wall time.

In [ ]:
phase6_log = run_stage('04_multibranch_matched_budget', ['scripts/benchmark_multibranch_matched_budget.jl'], 'PHASE6_MATCHED_BUDGET=PASS')

## 7. Full regression suite

In [ ]:
if RUN_FULL_TESTS:
    tests_log = run_stage('05_full_tests', ['test/runtests.jl'], 'Test Summary:')
    full_tests_status = 'PASS'
else:
    full_tests_status = 'SKIPPED'
    print('Full tests skipped by RUN_FULL_TESTS=False')

## 8. Final acceptance summary

In [ ]:
summary = {
    'foundation': 'PASS' if 'FOUNDATION_VERIFICATION=PASS' in foundation_log else 'FAIL',
    'sage_gradient': 'PASS' if 'SAGE_GRADIENT_VERIFICATION=PASS' in sage_log else 'FAIL',
    'discontinuity_learning': 'PASS' if 'DISCONTINUITY_LEARNING_GO_NO_GO=PASS' in learning_log else 'FAIL',
    'multibranch_matched_budget': 'PASS' if 'PHASE6_MATCHED_BUDGET=PASS' in phase6_log else 'FAIL',
    'full_tests': full_tests_status,
    'git_commit': subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip(),
    'environment': environment,
    'stages': STAGE_RESULTS,
    'logs': str(LOG_DIR),
}
(LOG_DIR / 'final_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
archive = shutil.make_archive(str(WORK / 'sage_reproduction_bundle'), 'zip', root_dir=LOG_DIR)
print(json.dumps(summary, indent=2))
print('Downloadable log bundle:', archive)
try:
    from IPython.display import FileLink, display
    display(FileLink(archive))
except ImportError:
    pass
assert all(summary[key] == 'PASS' for key in ('foundation', 'sage_gradient', 'discontinuity_learning', 'multibranch_matched_budget'))
assert not RUN_FULL_TESTS or summary['full_tests'] == 'PASS'